# Phân tích Lợi nhuận và Hành vi Khách hàng - Đội NHLBike
Notebook này tái hiện các biểu đồ phân tích dữ liệu khám phá (EDA), các chẩn đoán nguyên nhân, và dự báo tệp khách hàng như đã trình bày trong báo cáo của đội **NHLBike** cho Datathon 2026.

## Cấu trúc nội dung:
1. **Hình 1**: Lợi nhuận gộp trung bình theo tháng & Tổng lợi nhuận gộp theo năm (2013-2022).
2. **Hình 2**: Tỷ lệ chiết khấu trung bình theo tháng giữa năm chẵn và năm lẻ.
3. **Hình 3**: Quy mô khách hàng mới/hoạt động & Heatmap tỷ lệ giữ chân (Cohort Retention).
4. **Hình 4**: Dự báo lượng khách hàng mới theo năm (2013-2024).
5. **Hình 5**: Lòng trung thành và tần suất mua hàng theo khu vực địa lý.

*Lưu ý: Các biểu đồ sau khi vẽ sẽ tự động được lưu vào thư mục `../reports/figures/` để phục vụ biên soạn tài liệu báo cáo chuyên nghiệp.*

In [ ]:
import os
os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib-cache')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.style.use('ggplot')
sns.set_theme(style="whitegrid")
sns.set_palette("viridis")

# Cấu hình đường dẫn dữ liệu và thư mục lưu biểu đồ
DATA_DIR = '../data/raw/'
FIG_DIR = '../reports/figures/'
os.makedirs(FIG_DIR, exist_ok=True)
print("Môi trường đã sẵn sàng. Thư mục lưu biểu đồ:", FIG_DIR)

## 0. Tải dữ liệu vào bộ nhớ
Chúng ta tải các bảng dữ liệu cần thiết cho EDA bao gồm: `sales.csv`, `orders.csv`, `order_items.csv`, `customers.csv`, `geography.csv`, `promotions.csv`, `returns.csv`.

In [ ]:
sales = pd.read_csv(DATA_DIR + 'sales.csv', parse_dates=['Date'])
orders = pd.read_csv(DATA_DIR + 'orders.csv', parse_dates=['order_date'])
order_items = pd.read_csv(DATA_DIR + 'order_items.csv')
customers = pd.read_csv(DATA_DIR + 'customers.csv', parse_dates=['signup_date'])
geo = pd.read_csv(DATA_DIR + 'geography.csv')
promos = pd.read_csv(DATA_DIR + 'promotions.csv', parse_dates=['start_date', 'end_date'])
returns = pd.read_csv(DATA_DIR + 'returns.csv')

print("Dữ liệu tải thành công!")
print(f"- Sales: {sales.shape[0]} dòng từ {sales['Date'].min().date()} đến {sales['Date'].max().date()}")
print(f"- Orders: {orders.shape[0]} dòng từ {orders['order_date'].min().date()} đến {orders['order_date'].max().date()}")
print(f"- Customers: {customers.shape[0]} khách hàng")

## 1. Phân tích Lợi nhuận gộp (Hình 1)
- **Lợi nhuận gộp (Gross Profit)** = Revenue - COGS.
- Chúng ta phân tích lợi nhuận theo hai lát cắt:
  1. **Bên trái**: Lợi nhuận gộp trung bình theo tháng nhằm phân tích tính mùa vụ (Seasonality).
  2. **Bên phải**: Tổng lợi nhuận gộp theo năm (2013-2022) nhằm chỉ ra xu hướng dài hạn và chu kỳ chẵn-lẻ.

In [ ]:
# Tính lợi nhuận gộp
sales['GrossProfit'] = sales['Revenue'] - sales['COGS']
sales['year'] = sales['Date'].dt.year
sales['month'] = sales['Date'].dt.month

# Phân tích theo năm (lọc các năm đầy đủ từ 2013 đến 2022)
annual_gp = sales[(sales['year'] >= 2013) & (sales['year'] <= 2022)].groupby('year')['GrossProfit'].sum() / 1e9 # Đơn vị: Tỷ đồng

# Phân tích theo tháng (Lợi nhuận gộp trung bình của các tháng)
monthly_sums = sales[(sales['year'] >= 2013) & (sales['year'] <= 2022)].groupby(['year', 'month'])['GrossProfit'].sum().reset_index()
monthly_gp_avg = monthly_sums.groupby('month')['GrossProfit'].mean() / 1e6 # Đơn vị: Triệu đồng

# Trực quan hóa
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Biểu đồ bên trái: Lợi nhuận gộp trung bình theo tháng
sns.barplot(x=monthly_gp_avg.index, y=monthly_gp_avg.values, ax=axes[0], color='teal')
axes[0].set_title('Lợi nhuận gộp trung bình theo tháng (2013–2022)', fontsize=13, fontweight='bold', pad=15)
axes[0].set_xlabel('Tháng', fontsize=11)
axes[0].set_ylabel('Lợi nhuận gộp (Triệu VND)', fontsize=11)
for container in axes[0].containers:
    axes[0].bar_label(container, fmt='%.1f', padding=3, fontsize=9)

# Biểu đồ bên phải: Tổng lợi nhuận gộp theo năm
sns.lineplot(x=annual_gp.index, y=annual_gp.values, ax=axes[1], marker='o', color='navy', linewidth=2.5, markersize=8)
axes[1].set_title('Tổng lợi nhuận gộp theo năm (2013–2022)', fontsize=13, fontweight='bold', pad=15)
axes[1].set_xlabel('Năm', fontsize=11)
axes[1].set_ylabel('Lợi nhuận gộp (Tỷ VND)', fontsize=11)
axes[1].set_xticks(annual_gp.index)
for x, y in zip(annual_gp.index, annual_gp.values):
    axes[1].annotate(f"{y:.3f}B", (x, y), textcoords="offset points", xytext=(0,10), ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig1_gross_profit.png'), dpi=300)
plt.show()

## 2. Phân tích Tỷ lệ chiết khấu (Hình 2)
- Báo cáo chỉ ra chu kỳ chẵn lẻ xuất phát từ chính sách chiết khấu: **Năm lẻ khuyến mãi nhiều hơn và chiết khấu sâu hơn**, đặc biệt là tháng 8 (28.6% do Urban Blowout) và tháng 12 (19.9% do Year-End Sale).
- Để tái hiện chính xác con số này, ta thực hiện:
  1. Ghép bảng chi tiết đơn hàng `order_items` với đơn hàng `orders` và bảng khuyến mãi `promotions`.
  2. Tính mức chiết khấu định danh (nominal discount) của từng dòng giao dịch. Nếu giao dịch không có khuyến mãi, chiết khấu mặc định là `0.0`.
  3. Tính mức chiết khấu trung bình theo tháng, chia thành 2 nhóm: Năm Chẵn (Even Years) và Năm Lẻ (Odd Years).

In [ ]:
# Merge giao dịch với đơn hàng để lấy ngày tháng năm
df_items = pd.merge(order_items, orders, on='order_id')
df_items['year'] = df_items['order_date'].dt.year
df_items['month'] = df_items['order_date'].dt.month

# Merge tiếp với promotions để lấy giá trị khuyến mãi định danh
df_promo = pd.merge(df_items, promos, on='promo_id', how='left')
df_promo['nominal_disc'] = df_promo['discount_value'].fillna(0)

# Phân tách năm chẵn lẻ
df_promo['is_odd_year'] = df_promo['year'] % 2 != 0

# Tính tỷ lệ chiết khấu trung bình theo tháng
discount_trends = df_promo.groupby(['is_odd_year', 'month'])['nominal_disc'].mean().reset_index()

# Vẽ biểu đồ so sánh chẵn lẻ
plt.figure(figsize=(12, 6))
odd_data = discount_trends[discount_trends['is_odd_year'] == True]
even_data = discount_trends[discount_trends['is_odd_year'] == False]

plt.plot(odd_data['month'], odd_data['nominal_disc'], marker='o', color='darkorange', linewidth=2.5, markersize=8, label='Năm Lẻ (Odd Years)')
plt.plot(even_data['month'], even_data['nominal_disc'], marker='s', color='dodgerblue', linewidth=2, markersize=7, label='Năm Chẵn (Even Years)')

# Highlight các điểm T8 và T12 của năm lẻ
t8_val = odd_data[odd_data['month'] == 8]['nominal_disc'].values[0]
t12_val = odd_data[odd_data['month'] == 12]['nominal_disc'].values[0]

plt.annotate(f"T8: {t8_val:.1f}%", (8, t8_val), textcoords="offset points", xytext=(-20,15), 
             arrowprops=dict(arrowstyle="->", color='red', lw=1.5), fontsize=10, fontweight='bold', color='red')
plt.annotate(f"T12: {t12_val:.1f}%", (12, t12_val), textcoords="offset points", xytext=(-35,-20), 
             arrowprops=dict(arrowstyle="->", color='red', lw=1.5), fontsize=10, fontweight='bold', color='red')

plt.title('Tỷ lệ chiết khấu trung bình theo tháng: Năm chẵn vs Năm lẻ', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Tháng', fontsize=12)
plt.ylabel('Tỷ lệ chiết khấu định danh trung bình (%)', fontsize=12)
plt.xticks(range(1, 13))
plt.legend(frameon=True, fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig2_discount_rates.png'), dpi=300)
plt.show()

## 3. Phân tích Nền tảng khách hàng (Hình 3)
- Tệp khách hàng đang bị **suy giảm nghiêm trọng** ở cả hai khía cạnh đầu vào (thu hút mới) và đầu ra (giữ chân khách hàng cũ).
  1. **Khách hàng mới (New Customers)**: Được định nghĩa là những khách hàng thực hiện đơn hàng đầu tiên trong năm tương ứng.
  2. **Khách hàng hoạt động (Active Customers)**: Số lượng khách hàng duy nhất thực hiện ít nhất một giao dịch trong năm.
  3. **Cohort Retention Heatmap**: Theo dõi tỷ lệ giữ chân khách hàng từ cohort năm đăng ký đơn hàng đầu tiên qua các năm sau đó.

In [ ]:
# 3.1. Tính số lượng khách hàng mới và khách hàng hoạt động
orders['year'] = orders['order_date'].dt.year

# Tìm năm giao dịch đầu tiên của từng khách hàng
first_orders = orders.groupby('customer_id')['order_date'].min().reset_index()
first_orders['cohort_year'] = first_orders['order_date'].dt.year

# Đếm số khách mới theo năm
annual_new = first_orders.groupby('cohort_year')['customer_id'].count()

# Đếm số khách hoạt động theo năm
annual_active = orders.groupby('year')['customer_id'].nunique()

# 3.2. Tính toán Cohort Retention Matrix
df_ret = pd.merge(orders, first_orders[['customer_id', 'cohort_year']], on='customer_id')

cohort_matrix = []
cohort_sizes = {}

for cohort in sorted(df_ret['cohort_year'].unique()):
    cohort_users = df_ret[df_ret['cohort_year'] == cohort]
    total_users = cohort_users['customer_id'].nunique()
    cohort_sizes[cohort] = total_users
    
    # Số lượng khách hoạt động trong từng năm tiếp theo
    active_by_year = cohort_users.groupby('year')['customer_id'].nunique()
    
    row = []
    for yr in range(cohort, 2023):
        rate = active_by_year.get(yr, 0) / total_users
        row.append(rate)
    row += [np.nan] * (2023 - cohort - len(row))
    cohort_matrix.append(row)

# Tạo ma trận hiển thị
years_labels = sorted(df_ret['cohort_year'].unique())
retention_df = pd.DataFrame(cohort_matrix, index=years_labels, columns=[f"Y{i}" for i in range(len(years_labels))])

# Trực quan hóa Hình 3
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Bên trái: Quy mô khách hàng mới vs khách hàng hoạt động
axes[0].plot(annual_new.index, annual_new.values / 1e3, marker='o', color='crimson', linewidth=2.5, label='Khách hàng mới (K)')
axes[0].plot(annual_active.index, annual_active.values / 1e3, marker='s', color='darkslategray', linewidth=2.5, label='Khách hàng hoạt động (K)')
axes[0].set_title('Biến động quy mô khách hàng mới & khách hoạt động (2013-2022)', fontsize=13, fontweight='bold', pad=15)
axes[0].set_xlabel('Năm', fontsize=11)
axes[0].set_ylabel('Số lượng khách hàng (Nghìn người)', fontsize=11)
axes[0].set_xticks(range(2013, 2023))
axes[0].legend(frameon=True, fontsize=10)
for x, y in zip(annual_new.index, annual_new.values):
    if x in [2013, 2022]:
        axes[0].annotate(f"{y/1e3:.1f}K", (x, y/1e3), textcoords="offset points", xytext=(0,10), ha='center', fontweight='bold', color='crimson')
for x, y in zip(annual_active.index, annual_active.values):
    if x in [2016, 2022]:
        axes[0].annotate(f"{y/1e3:.1f}K", (x, y/1e3), textcoords="offset points", xytext=(0,-15), ha='center', fontweight='bold', color='darkslategray')

# Bên phải: Heatmap tỷ lệ giữ chân khách hàng
sns.heatmap(retention_df, annot=True, fmt=".1%", cmap="YlGnBu", ax=axes[1], cbar=False)
axes[1].set_title('Tỷ lệ giữ chân khách hàng theo các thế hệ Cohort', fontsize=13, fontweight='bold', pad=15)
axes[1].set_xlabel('Năm hoạt động thứ (Y0 = Năm đầu tiên)', fontsize=11)
axes[1].set_ylabel('Cohort (Năm tham gia)', fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig3_customer_decay.png'), dpi=300)
plt.show()

## 4. Dự báo Khách hàng mới (Hình 4)
- Dự báo lượng khách hàng mới sẽ tiếp tục suy giảm về mức kỷ lục: **1,060 khách hàng (2023)** và **945 khách hàng (2024)**.
- Chúng ta vẽ đồ thị lịch sử kết hợp đường nét đứt biểu thị xu hướng dự báo tương lai cho giai đoạn 2023-2024 để cảnh báo ban lãnh đạo về việc nền tảng khách hàng đang sụt giảm nghiêm trọng.

In [ ]:
# Lấy chuỗi lịch sử khách hàng mới
hist_years = np.array(annual_new.index[1:]) # Bỏ qua 2012 vì là năm không trọn vẹn
hist_counts = np.array(annual_new.values[1:])

# Bổ sung dữ liệu dự báo cho 2023 và 2024
forecast_years = np.array([2023, 2024])
forecast_counts = np.array([1060, 945])

# Nối chuỗi để vẽ đường dự báo liền mạch
all_years = np.concatenate([hist_years, forecast_years])
all_counts = np.concatenate([hist_counts, forecast_counts])

plt.figure(figsize=(12, 6))
# Vẽ phần lịch sử
plt.plot(hist_years, hist_counts, marker='o', color='forestgreen', linewidth=2.5, label='Lịch sử (2013-2022)')
# Vẽ phần dự báo bằng nét đứt
plt.plot(all_years[-3:], all_counts[-3:], marker='o', linestyle='--', color='darkred', linewidth=2.5, label='Dự báo (2023-2024)')

# Thêm nhãn số liệu cho các điểm mốc chính, đặt lệch hướng để không chồng chữ
plt.annotate(f"2013: {hist_counts[0]:,}", (2013, hist_counts[0]), textcoords="offset points", xytext=(0,10), ha='center', fontweight='bold')
plt.annotate(f"2022: {hist_counts[-1]:,}", (2022, hist_counts[-1]), textcoords="offset points", xytext=(-48,26), ha='right', fontweight='bold', color='forestgreen')
plt.annotate(f"2023: {forecast_counts[0]:,}", (2023, forecast_counts[0]), textcoords="offset points", xytext=(-8,24), ha='right', fontweight='bold', color='darkred')
plt.annotate(f"2024: {forecast_counts[1]:,}", (2024, forecast_counts[1]), textcoords="offset points", xytext=(28,20), ha='left', fontweight='bold', color='darkred')

plt.title('Dự báo lượng khách hàng mới theo Năm (2013 - 2024)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Năm', fontsize=12)
plt.ylabel('Số lượng khách hàng mới (người)', fontsize=12)
plt.xticks(all_years)
plt.ylim(bottom=0, top=max(hist_counts) * 1.08)
plt.legend(frameon=True, fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig4_new_customer_forecast.png'), dpi=300)
plt.show()


## 5. Phân tích Lòng trung thành theo Khu vực (Hình 5)
- Để đưa ra giải pháp định lượng, nhóm đề xuất chiến dịch **"The West Loyalists"** nhằm tập trung khai thác thị trường nòng cốt phía Tây.
- Chúng ta phân tích lòng trung thành của khách hàng qua hai chỉ số:
  1. **Trung vị khoảng cách giữa các đơn hàng (Median Recency Days)**: Thời gian trung bình giữa 2 đơn hàng liên tiếp của một khách hàng.
  2. **Số lần quay lại trung bình (Average Orders per Customer)**: Số lượng đơn hàng trung bình một khách hàng đã đặt.

In [ ]:
# Ghép bảng khách hàng và địa lý để lấy khu vực (region)
cust_geo = pd.merge(customers, geo, on='zip')

# Tránh tính các đơn hàng bị trả để đo lường chính xác sức mua thực tế
valid_orders = orders[~orders['order_id'].isin(returns['order_id'])]
df_loyalty = pd.merge(valid_orders, cust_geo, on='customer_id')

# 1. Số lần mua hàng trung bình theo từng miền
orders_per_cust = df_loyalty.groupby(['region', 'customer_id'])['order_id'].count().reset_index()
avg_orders_by_region = orders_per_cust.groupby('region')['order_id'].mean()

# 2. Trung vị số ngày giữa các đơn hàng
df_sorted = df_loyalty.sort_values(by=['customer_id', 'order_date'])
df_sorted['prev_date'] = df_sorted.groupby('customer_id')['order_date'].shift(1)
df_sorted['days_diff'] = (df_sorted['order_date'] - df_sorted['prev_date']).dt.days
median_days_by_region = df_sorted.groupby('region')['days_diff'].median()

# Trực quan hóa Hình 5
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Subplot 1: Trung vị khoảng cách giữa các đơn hàng
sns.barplot(x=median_days_by_region.index, y=median_days_by_region.values, ax=axes[0], palette='coolwarm')
axes[0].set_title('Trung vị số ngày giữa các đơn hàng theo khu vực', fontsize=13, fontweight='bold', pad=15)
axes[0].set_xlabel('Khu vực', fontsize=11)
axes[0].set_ylabel('Số ngày (Càng ngắn càng tốt)', fontsize=11)
for container in axes[0].containers:
    axes[0].bar_label(container, fmt='%d ngày', padding=3, fontsize=10, fontweight='bold')

# Subplot 2: Số đơn hàng trung bình của một khách hàng
sns.barplot(x=avg_orders_by_region.index, y=avg_orders_by_region.values, ax=axes[1], palette='coolwarm')
axes[1].set_title('Số đơn hàng trung bình một khách hàng đặt theo khu vực', fontsize=13, fontweight='bold', pad=15)
axes[1].set_xlabel('Khu vực', fontsize=11)
axes[1].set_ylabel('Số lần mua hàng (lần)', fontsize=11)
for container in axes[1].containers:
    axes[1].bar_label(container, fmt='%.2f lần', padding=3, fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fig5_regional_loyalty.png'), dpi=300)
plt.show()

## 6. Tổng kết Đề xuất Hành động (Prescriptive Analytics)
Dựa trên các phân tích và biểu đồ trực quan phía trên, đội ngũ Data Science của **NHLBike** đề xuất 4 chiến dịch lớn:

1. **Chiến dịch "The West Loyalists"**:
   - **Phát hiện**: Khách hàng khu vực phía Tây có tần suất mua hàng cực cao (~11 lần so với ~6 lần của miền khác) và khoảng cách đặt hàng ngắn nhất (~90-95 ngày so với ~170 ngày).
   - **Hành động**: Tự động gửi email/tin nhắn ưu đãi mua lại ("Repurchase Voucher") vào ngày thứ 80 - 85 kể từ đơn hàng thành công gần nhất để đón đầu nhu cầu quay lại của nhóm này.
2. **Tái cấu trúc Khuyến mãi Urban Blowout**:
   - **Phát hiện**: Khuyến mãi tháng 8 (năm lẻ) làm sụt giảm lợi nhuận nghiêm trọng do áp dụng đại trà và chiết khấu định danh trung bình lên tới 28.6%.
   - **Hành động**: Chuyển đổi chiến dịch Urban Blowout từ công khai sang "Loyalty-only Event" (chỉ gửi mã giảm giá kín qua email cho khách hàng trung thành có lịch sử mua >5 đơn và tỉ lệ quay lại cao). Điều này giúp bảo vệ biên lợi nhuận gộp mà vẫn giữ chân được khách hàng cốt lõi.
3. **Tái định vị Thu hút Khách hàng mới (Acquisition)**:
   - **Phát hiện**: Lượng khách hàng mới giảm sụt 95% qua 9 năm trên tất cả các kênh.
   - **Hành động**: Đột phá kênh thu hút bằng cách kết hợp với Influencers ngách trong Streetwear/Outdoor và đẩy mạnh Social Commerce (TikTok/Instagram Shop) để trẻ hóa tệp khách hàng.
4. **Ràng buộc nghiệp vụ trong dự phóng (Domain Constraint)**:
   - **Mô hình hóa**: Áp dụng bộ lọc hậu xử lý kiểm soát tỉ lệ COGS/Revenue không vượt quá 130% để đảm bảo mô hình dự báo LightGBM luôn tuân thủ nguyên tắc thực tế của tài chính doanh nghiệp.